## ▶ Run Online — No Installation Needed

| Platform | Link |
|---|---|
| **Binder** (no account) | [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Piyushjhu/HELIX_Toolbox/main?labpath=examples%2F04_postprocessing_paper_plots.ipynb) |
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Piyushjhu/HELIX_Toolbox/blob/main/examples/04_postprocessing_paper_plots.ipynb) |
| **GitHub Codespaces** | [![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/Piyushjhu/HELIX_Toolbox) |

This notebook regenerates paper plots from existing SPADE summary CSVs. The setup cell below auto-detects the environment. If no summary CSV is available, run Examples 01–03 first to generate one.

# Example 4 — Post-Processing: Regenerate Paper Plots

This notebook shows how to use HELIX Toolbox's **post-processing mode** to
regenerate publication-quality figures from an existing SPADE analysis output
**without rerunning ALPSS or SPADE**.

This is useful when you want to:
- Tweak plot styles or axis limits
- Generate a figure that wasn't enabled in the original run
- Reproduce figures for a paper revision

Two approaches are shown:
1. **CLI post-processing mode** — via `helix_master_config.yml` with `enabled: true`
2. **Direct Python API** — calling `helix_paper_plots` functions directly

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys, subprocess

# ── Cloud / Online environment setup ──────────────────────────────────────
try:
    import google.colab
    _ENV = "colab"
except ImportError:
    _ENV = "binder" if os.environ.get("BINDER_SERVICE_HOST") else "local"

if _ENV in ("colab", "binder"):
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    os.environ["MPLBACKEND"] = "Agg"

if _ENV == "colab":
    REPO_ROOT = "/content/HELIX_Toolbox"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/Piyushjhu/HELIX_Toolbox.git",
                        REPO_ROOT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    os.path.join(REPO_ROOT, "requirements.txt")], check=False)
else:
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Environment: {_ENV} | REPO_ROOT: {REPO_ROOT}")
# ──────────────────────────────────────────────────────────────────────────

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# ── USER PATHS — edit these ────────────────────────────────────────────────
# Auto-detect SPADE output from Example 01; replace with your own path:
_auto_spade = os.path.join(REPO_ROOT, "examples", "figures",
                           "example_01_output", "SPADE_analysis")
SPADE_OUTPUT_DIR = _auto_spade if os.path.isdir(_auto_spade) \
                   else "/path/to/your/SPADE_analysis/"
OUTPUT_DIR = os.path.join(REPO_ROOT, "examples", "figures", "example_04_output")
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

summary_path  = os.path.join(SPADE_OUTPUT_DIR, "velocity_shots_summary.csv")
enhanced_path = os.path.join(SPADE_OUTPUT_DIR, "enhanced_spall_summary.csv")
print(f"SPADE_OUTPUT_DIR: {SPADE_OUTPUT_DIR}")
print("velocity_shots_summary.csv found:", os.path.exists(summary_path))
print("enhanced_spall_summary.csv found:", os.path.exists(enhanced_path))
if not os.path.exists(summary_path):
    print("\n⚠  Summary CSV not found. Run Examples 01-03 first to generate SPADE outputs.")

## Approach A: CLI post-processing mode

Set `post_processing_config.enabled: true` in the config and run the CLI.
ALPSS and SPADE are completely skipped — only plots are generated.

In [ ]:
import subprocess
from helix_analysis_toolbox import save_config_to_file

cfg = {
    "cli_settings": {
        "input_dir": None, "input_files": None, "input_pattern": "*.csv",
        "output_dir": OUTPUT_DIR, "param_folder": None,
        "analysis_mode": "both", "spade_mode": "auto",
        "spade_input_files": None, "spade_input_dir": None,
        "spade_input_pattern": "*--vel-smooth-with-uncert.csv",
    },
    "alpss_config": {},
    "spade_config": {
        "skip_unknown_material_traces": False,
        "experiment_hel_detection": True,
    },
    "post_processing_config": {
        "enabled": True,                      # <-- activates post-processing mode
        "spade_output_dir": SPADE_OUTPUT_DIR,
        "plots": {
            "hel_vs_peak_velocity":        True,
            "hel_vs_hel_strain_rate":      True,
            "hel_vs_laser_energy":         True,
            "shock_stress_vs_laser_energy":  True,
            "shock_stress_vs_waveplate_angle": True,
            "shock_stress_vs_peak_velocity": True,
            "laser_energy_vs_waveplate_angle": True,
            "flyer_row_column_peak_velocity_heatmap": True,
            "flyer_row_column_pair_peak_velocity": True,
            "flyer_row_column_pair_peak_velocity_by_material": True,
            "peak_velocity_pattern_analysis": True,
            "row_column_vs_peak_shock_stress": True,
        },
    },
    "material_properties": {
        "Cu":    {"density": 8960.0, "bulk_wave_speed": 3950.0, "C0": 3950.0, "C_L": 4700.0},
        "Zn":    {"density": 7140.0, "bulk_wave_speed": 3700.0, "C0": 3700.0, "C_L": 4200.0},
        "Brass": {"density": 8520.0, "bulk_wave_speed": 3800.0, "C0": 3800.0, "C_L": 4500.0},
        "Al":    {"density": 2700.0, "bulk_wave_speed": 5240.0, "C0": 5240.0, "C_L": 6000.0},
    },
}

cfg_path = os.path.join(OUTPUT_DIR, "postproc.yml")
save_config_to_file(cfg, cfg_path)

result = subprocess.run(
    [sys.executable, os.path.join(REPO_ROOT, "helix_cli_runner.py"),
     "--config", cfg_path],
    capture_output=False, text=True,
)
print("\nExit code:", result.returncode)

## Approach B: Python API — call helix_paper_plots directly

This gives finer control: you can load the data yourself, filter it,
and call individual plot functions.

In [ ]:
from helix_paper_plots import (
    apply_data_to_viz_poster_style,
    generate_spall_vs_strain_rate_plot,
    generate_spall_vs_strain_rate_by_material_subplots,
    generate_spall_vs_shock_stress_plot,
)

if os.path.exists(enhanced_path):
    df = pd.read_csv(enhanced_path)
    source = enhanced_path
elif os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    source = summary_path
else:
    df = None
    print("No summary CSV found — update SPADE_OUTPUT_DIR above.")

if df is not None:
    print(f"Loaded {len(df)} rows from {source}")
    print("Columns:", list(df.columns[:10]), "...")

In [ ]:
if df is not None:
    def progress(msg):
        print(" ", msg)

    apply_data_to_viz_poster_style()

    # Spall strength vs strain rate (all materials combined)
    generate_spall_vs_strain_rate_plot(
        summary_df=df,
        spade_output_dir=OUTPUT_DIR,
        progress_callback=progress,
    )

    # Per-material subplots
    generate_spall_vs_strain_rate_by_material_subplots(
        summary_df=df,
        spade_output_dir=OUTPUT_DIR,
        progress_callback=progress,
    )

    print("\nPlots saved to:", OUTPUT_DIR)

## Display the generated figures

In [ ]:
import glob as _glob
from IPython.display import Image, display as ipy_display

pngs = sorted(_glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
print(f"Found {len(pngs)} PNG files in {OUTPUT_DIR}")
for p in pngs[:6]:   # show up to 6
    print(os.path.basename(p))
    ipy_display(Image(p, width=800))

## Available helix_paper_plots functions

| Function | What it generates |
|----------|------------------|
| `generate_spall_vs_strain_rate_plot` | Spall strength vs strain rate (all materials) |
| `generate_spall_vs_strain_rate_by_material_subplots` | Per-material subplot grid |
| `generate_spall_vs_shock_stress_plot` | Spall strength vs shock stress |
| `generate_spall_vs_shock_stress_by_material_subplots` | Per-material shock-stress subplots |
| `generate_peak_velocity_vs_time_plot` | Peak velocity stability over time |
| `generate_laser_energy_vs_time_plot` | Laser energy stability over time |
| `generate_laser_energy_stability_table` | Summary statistics table (CSV) |
| `generate_all_plots_from_summary_files` | Run all of the above in one call |

All functions accept a `progress_callback` argument (any callable that takes
a string) so you can hook them into a GUI progress bar or logging system.